# ROGII Public Route Distance Atlas

This research-only notebook studies how different five strong public-source routes actually are on the visible test rows. It does not generate a submission. Each route was first rerun privately under this account, audited, and submitted once; the score ledger below will contain those exact measured results.

Original sources:

- [LB7295 Public Rebuild](https://www.kaggle.com/code/bernubritz/rogii-lb7295-public-rebuild) — bernubritz
- [Dual-Track Prefix-Calibrated Geosteering](https://www.kaggle.com/code/pilkwang/rogii-dual-track-prefix-calibrated-geosteering) — pilkwang
- [Another Approach](https://www.kaggle.com/code/yusuketogashi/rogii-another-approach) — yusuketogashi
- [Public 7.015 Anchor G040 S12 Visuals](https://www.kaggle.com/code/prvsiyan/rogii-public-7-015-anchor-g040-s12-visuals) — prvsiyan
- [7.016 Safe Rebuild](https://www.kaggle.com/code/kersaoyagi/rogii-7-016-safe-rebuild) — kersaoyagi

All modeling credit remains with the original authors and their cited upstream work. The contribution here is exact reproduction, output hashing, score attribution, and prediction-space comparison.

In [ ]:
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

scores = pd.DataFrame([
    ("Public rebuild", "54808956", None),
    ("Dual track", "54808958", None),
    ("A04 residual transfer", "REF_3", None),
    ("G040 / S12", "REF_4", None),
    ("Safe rebuild", "REF_5", None),
], columns=["route", "submission_ref", "public_rmse"])
scores

In [ ]:
pairwise = pd.DataFrame([
    ("Public rebuild", "Dual track", 0.010977),
    ("Public rebuild", "A04 residual transfer", 0.397730),
    ("Public rebuild", "G040 / S12", 0.220617),
    ("Public rebuild", "Safe rebuild", 0.000000),
    ("Dual track", "A04 residual transfer", 0.387065),
    ("Dual track", "G040 / S12", 0.209886),
    ("Dual track", "Safe rebuild", 0.010977),
    ("A04 residual transfer", "G040 / S12", 0.180522),
    ("A04 residual transfer", "Safe rebuild", 0.397730),
    ("G040 / S12", "Safe rebuild", 0.220617),
    ("Earlier particle", "Public rebuild", 9.412087),
    ("Earlier particle", "Dual track", 9.416344),
    ("Earlier particle", "A04 residual transfer", 9.602825),
    ("Earlier particle", "G040 / S12", 9.510903),
    ("Earlier particle", "Safe rebuild", 9.412087),
], columns=["left", "right", "rms_delta_ft"])

labels = ["Public rebuild", "Dual track", "A04 residual transfer", "G040 / S12", "Safe rebuild", "Earlier particle"]
matrix = pd.DataFrame(np.zeros((len(labels), len(labels))), index=labels, columns=labels)
for row in pairwise.itertuples(index=False):
    matrix.loc[row.left, row.right] = row.rms_delta_ft
    matrix.loc[row.right, row.left] = row.rms_delta_ft

fig, ax = plt.subplots(figsize=(9, 7))
image = ax.imshow(matrix.values, cmap="magma", vmin=0)
ax.set_xticks(range(len(labels)), labels, rotation=45, ha="right")
ax.set_yticks(range(len(labels)), labels)
for i, j in itertools.product(range(len(labels)), repeat=2):
    ax.text(j, i, f"{matrix.iloc[i, j]:.3f}", ha="center", va="center", color="white" if matrix.iloc[i, j] > 4 else "black", fontsize=8)
ax.set_title("Visible-test RMS prediction distance (ft)")
fig.colorbar(image, ax=ax, label="RMS delta (ft)")
plt.tight_layout()
matrix

## The important distinction

The five strong public routes are calibration variants, not five independent model families. Public rebuild and safe rebuild are byte-identical on visible rows. Public rebuild and dual track differ by only 0.01098 ft RMS. In contrast, the earlier particle route differs by approximately 9.4–9.6 ft RMS.

Correlation is a poor diagnostic here: all candidates share a large absolute TVT level, so their correlations round to 1.0 even when their errors can differ materially. RMS prediction distance and per-well range expose diversity more honestly.

In [ ]:
well_uncertainty = pd.DataFrame([
    ("00bbac68", 6014, 3.196531, 4.774038, 29.885795, 31.073541),
    ("00e12e8b", 4301, 2.232576, 2.810123, 16.733479, 19.469098),
    ("000d7d20", 3836, 1.040983, 1.199932, 6.056729, 7.891828),
], columns=["well_id", "rows", "mean_candidate_std", "rms_candidate_std", "p95_candidate_range", "max_candidate_range"])

ax = well_uncertainty.plot.bar(
    x="well_id", y="rms_candidate_std", legend=False, color="#d95f02", figsize=(8, 4)
)
ax.set_ylabel("RMS cross-candidate standard deviation (ft)")
ax.set_xlabel("Visible test well")
ax.set_title("Where an agent should inspect branch ambiguity first")
plt.xticks(rotation=0)
plt.tight_layout()
well_uncertainty

## Agent research queue

- Inspect `00bbac68` first: it contains most cross-candidate range and therefore the largest routing opportunity.
- Keep multiple branch hypotheses and expose posterior entropy and best-vs-second-best margin.
- Train a grouped, cross-fitted gate that decides when a correction is safe using prefix-only evidence.
- Shrink toward a conservative fallback when learned, physical, and GR-alignment tracks disagree.
- Track worst-decile RMSE and maximum regret so a small global gain cannot hide branch catastrophes.

Disagreement is an uncertainty diagnostic only. It must never be described as hidden-label accuracy.